In [42]:
from InputData import *

instance_filename = "AnzahlAuftraege_NEW_10/Construction_a10_o107_m5_an57_ar12.json"
#instance_filename = "AnzahlAuftraege_NEW_50/Construction_a50_o625_m30_an258_ar61.json"
#instance_filename = "Construction_a1_o12_m3_an5_ar3_reduced.json"

data = InputData(instance_filename)


In [50]:
# 2a. Sets
M = list()
W_m = dict()
N_m = dict()
for machine in data.machines:
    M.append(machine.id)
    W_m[machine.id] = [int(driver) for driver in machine.default_drivers]
    N_m[machine.id] = list()
    for orderItem in data.order_items:
        if orderItem.machine_type == machine.type:
            N_m[machine.id].append(orderItem.id)

W = list()
N_w = dict() # ANNAHME: N_w ist die Menge der Bestellungen, die ein Arbeiter bearbeiten kann
for worker in data.workers:
    W.append(worker.personal_number)
    for orderItem in data.order_items:
        if orderItem.worker_qualifications == []:
            if worker.personal_number not in N_w:
                N_w[worker.personal_number] = list()
            N_w[worker.personal_number].append(orderItem.id)
        elif orderItem.worker_qualifications == worker.qualifications:
            if worker.personal_number not in N_w:
                N_w[worker.personal_number] = list()
            N_w[worker.personal_number].append(orderItem.id)
        

'''
A = list()
A_Class = list()
N_a = dict() # ANNAHME: N_a ist die Menge der Bestellungen, die ein Anbaugerät bearbeiten kann
for attachment in data.attachments:
    A.append(attachment.id)
    A_Class.append(attachment.type)
'''
    
N = list()
for orderItem in data.order_items:
    N.append(orderItem.id)

C = list()
N_c = dict()
for order in data.orders:
    C.append(order.site_number)
    N_c[order.site_number] = order.order_item_ids


start_date = data.start_date
end_date = data.end_date

O_t = dict()  # Tag an dem der Auftrag startet
O_t_start = dict()  # Startzeiten  
O_t_end = dict()  # Endzeiten
O_t_start_inverted = dict()  # Umgekehrtes O_t (Startzeiten)
O_t_end_inverted = dict()  # Umgekehrtes O_t_end (Endzeiten)

SECONDS_IN_A_DAY = 86400

for orderItem in data.order_items:
    orderID = orderItem.id 

    # Startzeit
    orderItem_start_date = orderItem.start_time
    delta_start = (orderItem_start_date - start_date)
    t_start = delta_start.total_seconds() / SECONDS_IN_A_DAY
    t_start_int = int(t_start)


    # O_t: Gruppiert nach Tagen
    if t_start_int not in O_t:
        O_t[t_start_int] = []
    O_t[t_start_int].append(orderID)
    
    
    # Startzeit
    if t_start not in O_t_start:
        O_t_start[t_start] = []
    O_t_start[t_start].append(orderID)
    
    # Invertiertes Dictionary O_t_start_inverted
    O_t_start_inverted[orderID] = t_start

    # Endzeit
    orderItem_end_date = orderItem.end_time
    delta_end = (orderItem_end_date - start_date)
    t_end = delta_end.total_seconds() / SECONDS_IN_A_DAY

    # O_t_end: Gruppiert nach Endzeit
    if t_end not in O_t_end:
        O_t_end[t_end] = []
    O_t_end[t_end].append(orderID)
    
    # Invertiertes Dictionary O_t_end_inverted
    O_t_end_inverted[orderID] = t_end



P_mn = dict()
S_mn = dict()

d_ab = data.transport_routes
d_wb = data.work_routes
d_ij = list()
d_wj = list()

for i in data.order_items:
    row = []
    for j in data.order_items:
        a = next((k for k,v in N_c.items() if str(i.id) in v))
        b = next((k for k,v in N_c.items() if str(j.id) in v))
        row.append(d_ab[a][b])
    d_ij.append(row)


for i in data.workers:
    row = []
    for j in data.order_items:
        a = next((k for k,v in N_c.items() if str(j.id) in v))
        row.append(d_wb[i.personal_number][a])
    d_wj.append(row)



SPEED = 1680 # Durchschnittliche Geschwindigkeit des Maschinentransports in 1680 km/Tag --> entspricht 70 km/h

for m in M:
    for n in N_m[m]:
        P_mn[m,n] = list()
        S_mn[m,n] = list()
        for i in N_m[m]:
            if n != i:
                start_time_n = O_t_start_inverted[n]
                end_time_n = O_t_end_inverted[n]
                start_time_i = O_t_start_inverted[i]
                end_time_i = O_t_end_inverted[i]


                if start_time_n >= end_time_i + d_ij[i][n] / SPEED: 
                    P_mn[m,n].append(i)

                if start_time_i > end_time_n + d_ij[n][i] / SPEED:
                    S_mn[m,n].append(i)

P_wn = dict()
S_wn = dict()

P_time = 9/24 # 9 Stunden Pausenzeit zwischen zwei Schichten --> 9/24 Tage

for w in W:
    for n in N_w[w]:
        P_wn[w,n] = list()
        S_wn[w,n] = list()
        for i in N_w[w]:
            if n != i:
                start_time_n = O_t_start_inverted[n]
                end_time_n = O_t_end_inverted[n]
                start_time_i = O_t_start_inverted[i]
                end_time_i = O_t_end_inverted[i]


                if start_time_n >= end_time_i + P_time:
                    P_wn[w,n].append(i)

                if start_time_i >= end_time_n + P_time:
                    S_wn[w,n].append(i)
        



day_difference = end_date - start_date
T_range = list(range(day_difference.days))


# 2b. Parameter

T = day_difference.days


S_Nmax = 5 # Maximal Anzahl an aufeinanderfolgenden Nachtschichten
S_max = 10 # Maximal Anzahl an Schichten im Zeitraum T_Smax
T_Smax = 14 # Zeitraum für S_max
T_Wmax = 40 # Maximale Arbeistzeit im Betrachtungszeitraum/Monat ?

t_o = list()
for orderItem in data.order_items:
    t_o.append(orderItem.duration)



print(S_wn)

{(0, 13): [15, 16, 103, 104, 105, 106], (0, 14): [16, 104, 105, 106], (0, 15): [105, 106], (0, 16): [106], (0, 93): [13, 14, 15, 16, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106], (0, 94): [13, 14, 15, 16, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106], (0, 95): [13, 14, 15, 16, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106], (0, 96): [13, 14, 15, 16, 98, 99, 100, 101, 102, 103, 104, 105, 106], (0, 97): [13, 14, 15, 16, 99, 100, 101, 102, 103, 104, 105, 106], (0, 98): [13, 14, 15, 16, 100, 101, 102, 103, 104, 105, 106], (0, 99): [13, 14, 15, 16, 101, 102, 103, 104, 105, 106], (0, 100): [14, 15, 16, 102, 103, 104, 105, 106], (0, 101): [15, 16, 103, 104, 105, 106], (0, 102): [16, 104, 105, 106], (0, 103): [105, 106], (0, 104): [106], (0, 105): [], (0, 106): [], (1, 13): [15, 16, 103, 104, 105, 106], (1, 14): [16, 104, 105, 106], (1, 15): [105, 106], (1, 16): [106], (1, 93): [13, 14, 15, 16, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106], (1, 94): [13, 14, 15, 16, 96,

In [63]:
print(S_mn)

{(0, 0): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 18, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106], (0, 1): [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106], (0, 2): [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 

In [51]:
S_wn

{(0, 13): [15, 16, 103, 104, 105, 106],
 (0, 14): [16, 104, 105, 106],
 (0, 15): [105, 106],
 (0, 16): [106],
 (0, 93): [13,
  14,
  15,
  16,
  95,
  96,
  97,
  98,
  99,
  100,
  101,
  102,
  103,
  104,
  105,
  106],
 (0, 94): [13, 14, 15, 16, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106],
 (0, 95): [13, 14, 15, 16, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106],
 (0, 96): [13, 14, 15, 16, 98, 99, 100, 101, 102, 103, 104, 105, 106],
 (0, 97): [13, 14, 15, 16, 99, 100, 101, 102, 103, 104, 105, 106],
 (0, 98): [13, 14, 15, 16, 100, 101, 102, 103, 104, 105, 106],
 (0, 99): [13, 14, 15, 16, 101, 102, 103, 104, 105, 106],
 (0, 100): [14, 15, 16, 102, 103, 104, 105, 106],
 (0, 101): [15, 16, 103, 104, 105, 106],
 (0, 102): [16, 104, 105, 106],
 (0, 103): [105, 106],
 (0, 104): [106],
 (0, 105): [],
 (0, 106): [],
 (1, 13): [15, 16, 103, 104, 105, 106],
 (1, 14): [16, 104, 105, 106],
 (1, 15): [105, 106],
 (1, 16): [106],
 (1, 93): [13,
  14,
  15,
  16,
  95,
  96,
  97,
  98,
  9